# Semana 9 - Pytorch e Redes Neurais Artificiais

## Autor
Victor Bertolini de Sousa

Github: https://github.com/VictorBertolini/Onboarding-LIPAI

## Resumo Modelos Estudados

- MLP (Perceptron Multicamadas):

Rede neural do tipo feedforward composta por camadas de neurônios totalmente conectadas. Utiliza funções de ativação não lineares para aprender padrões complexos nos dados. É amplamente aplicada em problemas de classificação e regressão. Seu treinamento é feito por meio do algoritmo de backpropagation.

- CNN (Redes Convolucionais):

Projetadas para processar dados estruturados em forma de grade, como imagens. Utilizam camadas convolucionais com filtros que detectam padrões locais, como bordas e texturas. Também incluem camadas de pooling para reduzir dimensionalidade. São muito usadas em visão computacional e reconhecimento de imagens.

- RNN (Redes Recorrentes):

Indicadas para dados sequenciais, pois possuem conexões que permitem manter informações de estados anteriores. São usadas em tarefas como processamento de linguagem natural e previsão de séries temporais. Variantes como LSTM e GRU foram criadas para lidar melhor com dependências de longo prazo.

- Transformers:

Arquitetura moderna baseada em mecanismos de autoatenção, que permitem analisar relações entre todos os elementos da entrada simultaneamente. Diferente das RNNs, não processam dados de forma sequencial, o que melhora a eficiência. São amplamente utilizados em tarefas de linguagem natural.

## Explicação do Treinamento de Redes Neurais

O treinamento consiste em ajustar os pesos da rede para minimizar o erro entre as previsões e os valores reais.

Para isso, utiliza-se uma função de perda, que mede o quão errado o modelo está.

Em seguida, o algoritmo de backpropagation calcula como cada peso contribui para o erro, propagando essa informação de volta pela rede.

Por fim, os otimizadores atualizam os pesos de forma iterativa para melhorar o desempenho do modelo ao longo do tempo.

## Exercícios


#### 1 - Com base no material apresentado no notebook, o que é uma função de ativação, como a ReLU? Por que normalmente usamos funções de ativação entre as camadas?

A função de ativação é quem vai dizer para o neurônio a partir das entradas se ele deve ou não ativar e qual a sua força de ativação. Sem o uso dessas funções de ativação os modelos seriam como se fossem precisões lineares, mesmo tendo diversas camadas, eles permitem que a rede realmente aprenda

#### 2 - Explique o que cada uma das seguintes linhas de código faz e por que ela é necessária:

**● model.train()**

**● optimizer.step()**

**Além disso, explique:**

**Qual é a diferença fundamental entre os modos model.train() e model.eval()?**

`model_train()` -> Coloca o modelo em treinamento

`optimizer.step()` -> O Otimizador atualiza os pesos do modelo

`model.train()` -> ativa o modo de treinamento, com Dropout ligado e BatchNorm usando dados do batch atual

`model.eval()` -> ativa o modo de avaliação, desativando o Dropout e fazendo o BatchNorm usar estatísticas já aprendidas

## 3)

In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
titanic = sns.load_dataset('titanic')

feature_names = ['pclass', 'female', 'age', 'fare']
titanic['female'] = titanic['sex'].map({'male': 0, 'female': 1})
titanic.dropna(subset=feature_names, inplace=True)  #891 para 714

X = titanic[feature_names].to_numpy()
y = titanic['survived'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=123)

In [ ]:
class ClassBin(nn.Module):
    def __init__(self):
        super(ClassBin, self).__init__()
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        x = self.sigmoid(x)
        return x

model = ClassBin()
print(model)

In [ ]:
loss_fn = nn.BCELoss()
epochs = 100
batch_size = 32
learning_rate = 0.1

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Converter X e y para torch.Tensor
X_train = torch.Tensor(X_train)
y_train = torch.Tensor(y_train)
X_test = torch.Tensor(X_test)
y_test = torch.Tensor(y_test)

# Um Dataset de Tensores - Array [X, y]
train = TensorDataset(X_train, y_train)
train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)

test = TensorDataset(X_test, y_test)
test_loader = DataLoader(test, batch_size=batch_size, shuffle=True)

In [ ]:
for t in range(epochs):
    model.train() # Colocar o modelo em modo de treinamento (calcula os gradientes)

    # Batch Size
    for data in train_loader:
        # dar nome aos bois
        X = data[0]
        y = data[1]

        # Propagação (Feed Forward)
        y_pred = model(X)

        # Calcular erro usando a função-custo
        # y precisa virar um Tensor com tamanho (batch_size, 1)
        loss = loss_fn(y_pred, y.unsqueeze_(1))

        # Zera os gradientes antes da Retro-propagação (Backpropagation)
        model.zero_grad()

        # Retro-propagação (Backpropagation)
        loss.backward()

        # Atualização dos parâmetros
        optimizer.step()

    # Fim da Época
    print(f"""Época {t + 1}, Custo Treino: {round(loss.item(), 3)}""")

In [ ]:
model.eval() # coloca o modelo em modo de avaliação (sem calcular gradientes)

train_pred = model(X_train)
train_pred = train_pred.detach().apply_(lambda x : 1 if x > 0.5 else 0)
train_acc = torch.sum(train_pred.flatten() == y_train) / train_pred.size(0)

test_pred = model(X_test)
test_pred = test_pred.detach().apply_(lambda x : 1 if x > 0.5 else 0)
test_acc = torch.sum(test_pred.flatten() == y_test) / test_pred.size(0)

print(f"Acurácia de Treino: {train_acc}")
print('\n ---------------------------\n')
print(f"Acurácia de Teste: {test_acc}")

Pelo que pode ser observado, o modelo foi muito superior ao anteriormente visto, pois agora obteve 75% de acurácia nos testes, enquanto o outro entregava apenas 60%

## 4. Usando o modelo original do notebook:

**● Substitua o otimizador Adam por SGD**

**● Treine o modelo com o novo otimizador**

**● Observe o comportamento do custo (loss) durante o treinamento**

**● Compare a acurácia final com a obtida anteriormente**

In [ ]:
titanic = sns.load_dataset('titanic')

feature_names = ['pclass', 'female', 'age', 'fare']
titanic['female'] = titanic['sex'].map({'male': 0, 'female': 1})
titanic.dropna(subset=feature_names, inplace=True)  #891 para 714

X = titanic[feature_names].to_numpy()
y = titanic['survived'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=123)

In [ ]:
loss_fn = nn.BCELoss()
epochs = 100
batch_size = 32
learning_rate = 0.1

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
loss_fn = nn.BCELoss()
epochs = 100
batch_size = 32
learning_rate = 0.1

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Converter X e y para torch.Tensor
X_train = torch.Tensor(X_train)
y_train = torch.Tensor(y_train)
X_test = torch.Tensor(X_test)
y_test = torch.Tensor(y_test)

# Um Dataset de Tensores - Array [X, y]
train = TensorDataset(X_train, y_train)
train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)

test = TensorDataset(X_test, y_test)
test_loader = DataLoader(test, batch_size=batch_size, shuffle=True)

In [ ]:
for t in range(epochs):
    model.train() # Colocar o modelo em modo de treinamento (calcula os gradientes)

    # Batch Size
    for data in train_loader:
        # dar nome aos bois
        X = data[0]
        y = data[1]

        # Propagação (Feed Forward)
        y_pred = model(X)

        # Calcular erro usando a função-custo
        # y precisa virar um Tensor com tamanho (batch_size, 1)
        loss = loss_fn(y_pred, y.unsqueeze_(1))

        # Zera os gradientes antes da Retro-propagação (Backpropagation)
        model.zero_grad()

        # Retro-propagação (Backpropagation)
        loss.backward()

        # Atualização dos parâmetros
        optimizer.step()

    # Fim da Época
    print(f"""Época {t + 1}, Custo Treino: {round(loss.item(), 3)}""")

In [ ]:
model.eval() # coloca o modelo em modo de avaliação (sem calcular gradientes)

train_pred = model(X_train)
train_pred = train_pred.detach().apply_(lambda x : 1 if x > 0.5 else 0)
train_acc = torch.sum(train_pred.flatten() == y_train) / train_pred.size(0)

test_pred = model(X_test)
test_pred = test_pred.detach().apply_(lambda x : 1 if x > 0.5 else 0)
test_acc = torch.sum(test_pred.flatten() == y_test) / test_pred.size(0)

print(f"Acurácia de Treino: {train_acc}")
print('\n ---------------------------\n')
print(f"Acurácia de Teste: {test_acc}")

O modelo perdeu acurácia e demorou mais a convergir sem utilizar o otimizador adam

## 5)

In [1]:
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch import nn
import torch.nn.functional as F

In [3]:
class ClassBin(nn.Module):
    def __init__(self):
        super(ClassBin, self).__init__()
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        x = self.sigmoid(x)
        return x

model = ClassBin()
print(model)

ClassBin(
  (fc1): Linear(in_features=4, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=8, bias=True)
  (fc3): Linear(in_features=8, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [4]:
loss_fn = nn.BCELoss()
epochs = 100
batch_size = 512
learning_rate = 0.1

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [5]:
from torch.utils.data import TensorDataset, DataLoader

# Converter X e y para torch.Tensor
X_train = torch.Tensor(X_train)
y_train = torch.Tensor(y_train)
X_test = torch.Tensor(X_test)
y_test = torch.Tensor(y_test)

# Um Dataset de Tensores - Array [X, y]
train = TensorDataset(X_train, y_train)
train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)

test = TensorDataset(X_test, y_test)
test_loader = DataLoader(test, batch_size=batch_size, shuffle=True)

In [6]:
for t in range(epochs):
    model.train() # Colocar o modelo em modo de treinamento (calcula os gradientes)

    # Batch Size
    for data in train_loader:
        # dar nome aos bois
        X = data[0]
        y = data[1]

        # Propagação (Feed Forward)
        y_pred = model(X)

        # Calcular erro usando a função-custo
        # y precisa virar um Tensor com tamanho (batch_size, 1)
        loss = loss_fn(y_pred, y.unsqueeze_(1))

        # Zera os gradientes antes da Retro-propagação (Backpropagation)
        model.zero_grad()

        # Retro-propagação (Backpropagation)
        loss.backward()

        # Atualização dos parâmetros
        optimizer.step()

    # Fim da Época
    print(f"""Época {t + 1}, Custo Treino: {round(loss.item(), 3)}""")

Época 1, Custo Treino: 0.375
Época 2, Custo Treino: 1.824
Época 3, Custo Treino: 0.688
Época 4, Custo Treino: 1.377
Época 5, Custo Treino: 0.75
Época 6, Custo Treino: 0.679
Época 7, Custo Treino: 0.569
Época 8, Custo Treino: 0.674
Época 9, Custo Treino: 0.576
Época 10, Custo Treino: 0.62
Época 11, Custo Treino: 0.623
Época 12, Custo Treino: 0.595
Época 13, Custo Treino: 0.534
Época 14, Custo Treino: 0.627
Época 15, Custo Treino: 0.637
Época 16, Custo Treino: 0.481
Época 17, Custo Treino: 0.603
Época 18, Custo Treino: 0.494
Época 19, Custo Treino: 0.679
Época 20, Custo Treino: 0.467
Época 21, Custo Treino: 1.015
Época 22, Custo Treino: 0.576
Época 23, Custo Treino: 0.761
Época 24, Custo Treino: 0.711
Época 25, Custo Treino: 0.657
Época 26, Custo Treino: 0.659
Época 27, Custo Treino: 0.717
Época 28, Custo Treino: 0.548
Época 29, Custo Treino: 0.542
Época 30, Custo Treino: 0.585
Época 31, Custo Treino: 0.534
Época 32, Custo Treino: 0.605
Época 33, Custo Treino: 0.584
Época 34, Custo Trein

In [7]:
model.eval() # coloca o modelo em modo de avaliação (sem calcular gradientes)

train_pred = model(X_train)
train_pred = train_pred.detach().apply_(lambda x : 1 if x > 0.5 else 0)
train_acc = torch.sum(train_pred.flatten() == y_train) / train_pred.size(0)

test_pred = model(X_test)
test_pred = test_pred.detach().apply_(lambda x : 1 if x > 0.5 else 0)
test_acc = torch.sum(test_pred.flatten() == y_test) / test_pred.size(0)

print(f"Acurácia de Treino: {train_acc}")
print('\n ---------------------------\n')
print(f"Acurácia de Teste: {test_acc}")

Acurácia de Treino: 0.800000011920929

 ---------------------------

Acurácia de Teste: 0.7541899681091309


Com Batch size em 512 tenho acurácia de 75% nos testes

E com 4 eu tenho 60% nos testes, mostrando que, nesse contexto, o batch size de 512 foi melhor

## Reflexão

**1) Qual a importância da função de ativação em uma rede neural?**
As funções de ativação são essenciais porque introduzem não linearidade na rede. Sem elas, o modelo se comportaria como uma função linear, mesmo com várias camadas, limitando sua capacidade de aprender padrões complexos.

**2) Como a arquitetura da rede influencia o desempenho final?**
A arquitetura (número de camadas e neurônios) define a capacidade da rede de aprender. E uma mudança simples pode gerar desempenhos muito superiores ou muito inferiores

**3) O que muda ao trocar o otimizador?**
O otimizador define como os pesos são atualizados durante o treinamento. Diferentes otimizadores podem influenciar a velocidade de convergência e a qualidade da solução final.

**4) Como o tamanho do batch pode afetar o treinamento?**
O tamanho do batch afeta a forma como os gradientes são calculados. Batches pequenos tornam o treinamento mais ruidoso, mas podem ajudar na generalização. Batches grandes tornam o treinamento mais estável e eficiente computacionalmente, mas podem levar a uma pior generalização.

**5) Em quais situações uma rede mais simples pode ser mais adequada do que uma rede maior?**
Redes mais simples são mais adequadas quando há poucos dados, menor complexidade no problema ou necessidade de menor custo computacional. Elas também reduzem o risco de overfitting e são mais fáceis de treinar e interpretar.


